# 임포트 및 환경 설정

In [ ]:
import sys
import torch

sys.path.append('.')
from src.utils import set_seed, make_run_dir
from src.visualization import (plot_loss_history, 
                               plot_confusion_matrix, 
                               plot_model_comparison_pareto,
                                plot_model_comparison_bar, 
                                plot_weight_sweep, 
                                show_gradcam_grid, 
                                display_db_as_dataframe)
from src.engine import measure_inference_speed_with_gpu, measure_inference_speed_with_cpu, run_kfold_experiment
from src.dataset import get_image_paths_and_labels, val_transform

In [ ]:
set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 0  # Windows/Jupyter에서는 0이 가장 안정적
print(f"Device: {device} | num_workers: {NUM_WORKERS}")

run_dirs = make_run_dir('./outputs')

In [ ]:
DATA_DIR = './data/k-fold_data/train'
CLASS_NAMES = ['good', 'type1', 'type2', 'type3', 'type4', 'type5']

all_paths, all_labels_np = get_image_paths_and_labels(DATA_DIR, CLASS_NAMES)

In [ ]:
BAD_WEIGHT_LIST = [1.0, 1.5] # 테스트
MODELS_F2_LIST = []
MODELS_FPS_WITH_CPU = []
MODELS_FPS_WITH_GPU = []
MODEL_NAMES = ['ResNet-18', 'MobileNet-V2', 'VGG-18']

In [ ]:
# model_name, all_paths, all_labels_np,
# bad_weight_list, n_splits=5, num_epochs=50,
# batch_size=16, num_workers=NUM_WORKERS, device=device,
# run_dirs=run_dirs, early_stop_patience=10

resnet = run_kfold_experiment(
    model_name='resnet18',
    all_paths=all_paths,
    all_labels_np=all_labels_np,
    bad_weight_list=BAD_WEIGHT_LIST,
    n_splits=5,
    num_epochs=40,  # 디폴트는 50이었는데 일단 40만 해봅시다
    batch_size=16, 
    num_workers=NUM_WORKERS, 
    device=device,
    run_dirs=run_dirs, 
    early_stop_patience=10)

resnet_best_model = resnet[0]
resnet_weight_f2 = resnet[1]
resnet_best_train_loss = resnet[2]
resnet_best_val_loss = resnet[3]
y_true_test = resnet[4]
y_pred_test = resnet[5]
resnet_db = resnet[6]

In [ ]:
plot_weight_sweep(resnet_weight_f2, 'resnet18', run_dirs['figures'])

plot_loss_history(resnet_best_train_loss,
                  resnet_best_val_loss,
                  run_dirs['figures'],
                  model_name='resnet18')

resnet_test_metric = plot_confusion_matrix(y_true_test,
                                           y_pred_test,
                                           ['bad', 'good'],
                                           run_dirs['figures'],
                                           model_name='resnet18')
resnet_precision = resnet_test_metric[0]
resnet_recall = resnet_test_metric[1]
resnet_f2 = resnet_test_metric[2]
MODELS_F2_LIST.append(resnet_f2)

show_gradcam_grid(resnet_best_model,
                  val_transform,
                  'test',
                  device,
                  run_dirs['figures'],
                  model_name='resnet18')

resnet_gpu_fps, resnet_gpu_latency_ms = measure_inference_speed_with_gpu(resnet_best_model, 
                                                                         device, 
                                                                         input_size=(1, 3, 224, 224), 
                                                                         num_runs=100)
resnet_cpu_fps, resnet_cpu_latency_ms = measure_inference_speed_with_cpu(resnet_best_model, 
                                                                         input_size=(1, 3, 224, 224), 
                                                                         num_runs=100)
MODELS_FPS_WITH_CPU.append(resnet_cpu_fps)
MODELS_FPS_WITH_GPU.append(resnet_gpu_fps)

In [ ]:
display_db_as_dataframe(model_name='ResNet-18', db=resnet_db)

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt

# MODEL_NAMES = ['ResNet-18', 'MobileNet-V2', 'VGG-18']
# MODELS_F2_LIST = [0.9123, 0.8541, 0.9421]
# MODELS_FPS_WITH_CPU = [180, 200, 120]
# MODELS_FPS_WITH_GPU = [200, 220, 140]

plot_model_comparison_bar(model_names=MODEL_NAMES, 
                          f2_scores=MODELS_F2_LIST, 
                          fps_list=MODELS_FPS_WITH_CPU,
                          alpha=0.7,
                          save_dir='sda', 
                          device_label='CPU')

plot_model_comparison_bar(model_names=MODEL_NAMES, 
                          f2_scores=MODELS_F2_LIST, 
                          fps_list=MODELS_FPS_WITH_GPU,
                          alpha=0.7,
                          save_dir='sda', 
                          device_label='GPU')

plot_model_comparison_pareto(model_names=MODEL_NAMES,
                             f2_scores=MODELS_F2_LIST, 
                             fps_list=MODELS_F2_LIST, 
                             alpha=0.7, 
                             save_dir='asd', 
                             device_label='CPU')

plot_model_comparison_pareto(model_names=MODEL_NAMES,
                             f2_scores=MODELS_F2_LIST, 
                             fps_list=MODELS_F2_LIST, 
                             alpha=0.7, 
                             save_dir='asd', 
                             device_label='GPU')